# LEEDS

In [8]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, DeepGraphInfomax

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score, mean_squared_error
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from config import load

In [9]:

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [10]:
flows = pd.read_csv("./data/ODWP01EW_OA.csv")  # the travel-to-work data, 2021 Census
lookup_lsoa = pd.read_csv("./data/PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv") #the LSOA lookup file

C:\Users\jzvf437\AppData\Local\Temp\ipykernel_9108\2132882701.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup_lsoa = pd.read_csv("./data/PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv") #the LSOA lookup file


In [11]:
flows['Output Areas code'] = flows['Output Areas code'].astype(str).str.strip()
flows['OA of workplace code'] = flows['OA of workplace code'].astype(str).str.strip()
lookup_lsoa['oa21cd'] = lookup_lsoa['oa21cd'].astype(str).str.strip()
lookup_lsoa['lsoa21cd'] = lookup_lsoa['lsoa21cd'].astype(str).str.strip()

In [12]:
#creat a mapping dictionary from OA to LSOA
oa_to_lsoa=lookup_lsoa.set_index('oa21cd')['lsoa21cd'].to_dict()
#map the origin and destination OAs to LSOAs
flows['origin_lsoa'] = flows['Output Areas code'].map(oa_to_lsoa)
flows['workplace_lsoa'] = flows['OA of workplace code'].map(oa_to_lsoa)

In [13]:
# quick check: how many matched / unmatched 
print("origin_lsoa matched:", flows['origin_lsoa'].notna().sum(), 
      "unmatched:", flows['origin_lsoa'].isna().sum())
print("Workplace_LSOA matched:", flows['workplace_lsoa'].notna().sum(), 
      "unmatched:", flows['workplace_lsoa'].isna().sum())


# filter rows where both Origin_LSOA and Workplace_LSOA exist (no NaN)
flows_lsoa_complete = flows.dropna(subset=['origin_lsoa','workplace_lsoa']).copy()
print("Rows with both LSOAs present:", len(flows_lsoa_complete))

origin_lsoa matched: 10740405 unmatched: 780
Workplace_LSOA matched: 10480006 unmatched: 261179
Rows with both LSOAs present: 10479277


In [14]:
leeds_lsoa = lookup_lsoa[lookup_lsoa["ladnm"] == "Leeds"]["lsoa21cd"].unique()
print("Number of Leeds LSOAs:", len(leeds_lsoa))


#Filter flow data where origin and destination is in Leeds 

leeds_flows = flows_lsoa_complete[
    flows_lsoa_complete["origin_lsoa"].isin(leeds_lsoa) &
    flows_lsoa_complete["workplace_lsoa"].isin(leeds_lsoa)
]

print("Number of Leeds flows extracted:", len(leeds_flows))

Number of Leeds LSOAs: 488
Number of Leeds flows extracted: 108775


In [15]:
#taking out remote workers or unemployed

leeds_flows = leeds_flows[
    leeds_flows["Place of work indicator (4 categories) label"]
    != "Mainly working at or from home, No fixed place"
].copy()

In [16]:
len(leeds_flows)

106169

In [17]:
leeds_total_od = (
    leeds_flows
    .groupby(["origin_lsoa", "workplace_lsoa"])["Count"]
    .sum()
    .reset_index()
)

leeds_total_od.rename(columns={"Count": "flow"}, inplace=True)

print("Number of OD pairs:", len(leeds_total_od))
leeds_total_od.head()

Number of OD pairs: 55328


,origin_lsoa,workplace_lsoa,flow
0,E01011264,E01011265,17
1,E01011264,E01011266,1
2,E01011264,E01011267,1
3,E01011264,E01011268,5
4,E01011264,E01011269,3


In [18]:
# Step 1 — Remove self-loops and log transform
leeds_od = leeds_total_od[leeds_total_od['origin_lsoa'] != leeds_total_od['workplace_lsoa']].copy()
leeds_od['flow_log'] = np.log1p(leeds_od['flow'])

print(f"After removing self-loops: {len(leeds_od)}")
print(leeds_od.head())

After removing self-loops: 54907
  origin_lsoa workplace_lsoa  flow  flow_log
0   E01011264      E01011265    17  2.890372
1   E01011264      E01011266     1  0.693147
2   E01011264      E01011267     1  0.693147
3   E01011264      E01011268     5  1.791759
4   E01011264      E01011269     3  1.386294


In [19]:
import geopandas as gpd
# Read POIs (gpkg) and LSOA boundaries 
poi= gpd.read_file("./data/poi_uk.gpkg")       

# Replacing CSV with conversion to GeoDataFrame (using BNG eastings/northings) ---
LSOA_df = pd.read_csv("./data/Lower_layer_Super_Output_Areas_(December_2021)_Boundaries_EW_BFE_(V10)_and_RUC.csv")


In [20]:
# Basic mapping from main_category to broader group
category_to_group = {

    # FOOD & DRINK
    'restaurant': 'FOOD_DRINK',
    'pub': 'FOOD_DRINK',
    'bar': 'FOOD_DRINK',
    'cafe': 'FOOD_DRINK',
    'coffee_shop': 'FOOD_DRINK',
    'fast_food': 'FOOD_DRINK',
    'fish_and_chips_restaurant': 'FOOD_DRINK',
    'chilean_restaurant': 'FOOD_DRINK',

    # SHOPPING / RETAIL
    'shopping': 'RETAIL',
    'shopping_mall': 'RETAIL',
    'grocery_or_supermarket': 'RETAIL',
    'convenience_store': 'RETAIL',
    'department_store': 'RETAIL',

    # LEISURE / SPORT
    'active_life': 'LEISURE_SPORT',
    'gym': 'LEISURE_SPORT',
    'sports_club': 'LEISURE_SPORT',
    'stadium': 'LEISURE_SPORT',
    'cinema': 'LEISURE_SPORT',
    'museum': 'LEISURE_SPORT',
    'tourist_attraction': 'LEISURE_SPORT',

    # BEAUTY & PERSONAL
    'beauty_and_spa': 'BEAUTY_PERSONAL',
    'hair_salon': 'BEAUTY_PERSONAL',
    'nail_salon': 'BEAUTY_PERSONAL',
    'spa': 'BEAUTY_PERSONAL',

    # PROFESSIONAL / BUSINESS
    'professional_services': 'PROFESSIONAL',
    'lawyer': 'PROFESSIONAL',
    'accountant': 'PROFESSIONAL',
    'consulting_agency': 'PROFESSIONAL',

    # HEALTH
    'hospital': 'HEALTH',
    'doctor': 'HEALTH',
    'dentist': 'HEALTH',
    'pharmacy': 'HEALTH',
    'elder_care_planning': 'HEALTH',

    # EDUCATION
    'school': 'EDUCATION',
    'primary_school': 'EDUCATION',
    'secondary_school': 'EDUCATION',
    'college': 'EDUCATION',
    'university': 'EDUCATION',
    'parenting_classes': 'EDUCATION',

    # AUTO
    'tire_repair_shop': 'AUTO',
    'car_repair': 'AUTO',
    'car_dealer': 'AUTO',
    'petrol_station': 'AUTO',

    # NATURE / OUTDOOR
    'park': 'NATURE_OUTDOOR',
    'sand_dune': 'NATURE_OUTDOOR',
    'beach': 'NATURE_OUTDOOR',
    'forest': 'NATURE_OUTDOOR',

    # RELIGION
    'church_cathedral': 'RELIGION',
    'church': 'RELIGION',
    'mosque': 'RELIGION',
    'synagogue': 'RELIGION',
    'temple': 'RELIGION',

    # ACCOMMODATION
    'hotel': 'ACCOMMODATION',
    'motel': 'ACCOMMODATION',
    'campground': 'ACCOMMODATION',
    'guest_house': 'ACCOMMODATION',
    'holiday_rental_home': 'ACCOMMODATION',
}


In [21]:
# Map each POI to a broad group
poi['poi_group'] = poi['main_category'].map(category_to_group)

# Check how many got mapped
print(poi['poi_group'].value_counts(dropna=False).head(20))

# keep only mapped rows to start simple
poi_mapped = poi[~poi['poi_group'].isna()].copy()

poi_group
NaN                2003083
FOOD_DRINK          118348
PROFESSIONAL         87777
BEAUTY_PERSONAL      87072
RETAIL               61169
LEISURE_SPORT        48277
HEALTH               41098
ACCOMMODATION        31645
RELIGION             20421
AUTO                 10648
NATURE_OUTDOOR        7396
EDUCATION             4248
Name: count, dtype: int64


In [22]:
all_lsoas_leeds = pd.Index(pd.concat([leeds_od['origin_lsoa'],
                                        leeds_od['workplace_lsoa']]).unique())

print(f"Total unique LSOAs: {len(all_lsoas_leeds)}")

Total unique LSOAs: 488


In [23]:
# Pivot POI data to get counts per LSOA per category
poi_counts = (
    poi_mapped[poi_mapped['LSOA21CD'].isin(all_lsoas_leeds)]
    .groupby(['LSOA21CD', 'poi_group'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print(poi_counts.head())
print(f"Shape: {poi_counts.shape}")
print(f"POI categories: {poi_counts.columns.tolist()}")

poi_group   LSOA21CD  ACCOMMODATION  AUTO  BEAUTY_PERSONAL  EDUCATION  \
0          E01011264              0     0                0          0   
1          E01011265              0     2               16          0   
2          E01011266              1     1                3          0   
3          E01011267              0     0                0          0   
4          E01011268              0     0                0          0   

poi_group  FOOD_DRINK  HEALTH  LEISURE_SPORT  NATURE_OUTDOOR  PROFESSIONAL  \
0                   0       0              1               0             0   
1                  16      10              8               0             6   
2                   1       0              4               0             3   
3                   0       0              1               0             2   
4                   1       0              1               0             0   

poi_group  RELIGION  RETAIL  
0                 0       2  
1                 0       5  
2 

In [24]:
# Build feature matrix aligned to node index 
feature_df = pd.DataFrame({'LSOA21CD': list(all_lsoas_leeds)})

# Left join — missing LSOAs get 0 for all POI categories
feature_df = feature_df.merge(poi_counts, on='LSOA21CD', how='left')
feature_df = feature_df.fillna(0)

# Sanity check
assert len(feature_df) == len(all_lsoas_leeds), "Row count mismatch!"
assert list(feature_df['LSOA21CD']) == list(all_lsoas_leeds), "Node order mismatch!"

print(f"Feature matrix shape: {feature_df.shape}")  
print(f"LSOAs with all-zero POIs: {(feature_df.iloc[:, 1:].sum(axis=1) == 0).sum()}")
print(feature_df.head())

Feature matrix shape: (488, 12)
LSOAs with all-zero POIs: 6
    LSOA21CD  ACCOMMODATION  AUTO  BEAUTY_PERSONAL  EDUCATION  FOOD_DRINK  \
0  E01011264            0.0   0.0              0.0        0.0         0.0   
1  E01011265            0.0   2.0             16.0        0.0        16.0   
2  E01011266            1.0   1.0              3.0        0.0         1.0   
3  E01011267            0.0   0.0              0.0        0.0         0.0   
4  E01011268            0.0   0.0              0.0        0.0         1.0   

   HEALTH  LEISURE_SPORT  NATURE_OUTDOOR  PROFESSIONAL  RELIGION  RETAIL  
0     0.0            1.0             0.0           0.0       0.0     2.0  
1    10.0            8.0             0.0           6.0       0.0     5.0  
2     0.0            4.0             0.0           3.0       0.0     2.0  
3     0.0            1.0             0.0           2.0       1.0     0.0  
4     0.0            1.0             0.0           0.0       0.0     0.0  


In [25]:
# Save raw POI counts for gravity model (before normalisation)
poi_cols       = [c for c in feature_df.columns if c != 'LSOA21CD']
poi_raw_counts = feature_df[poi_cols].values.copy()

# Normalised 
poi_norm = np.log1p(poi_raw_counts)
poi_norm = (poi_norm - poi_norm.mean(axis=0)) / (poi_norm.std(axis=0) + 1e-8)

In [26]:
#  Load IMD data (LSOA level)
imd_data = pd.read_csv('./data/LSOA_IMD2025_OSGB1936_-7580747902701771840.csv')  

# Load LSOA population data
pop_data = pd.read_csv('./data/new_population_data.csv')

In [27]:
pop_data['Total'] = pop_data['Total'].astype(str).str.replace(',', '').astype(float)
pop_leeds = pop_data[pop_data['LSOA 2021 Code'].isin(all_lsoas_leeds)][['LSOA 2021 Code', 'Total']].copy()
pop_leeds = pop_leeds.rename(columns={'LSOA 2021 Code': 'LSOA21CD', 'Total': 'population'})

imd_leeds = imd_data[imd_data['LSOA Code'].isin(all_lsoas_leeds)][['LSOA Code', 'IMD Decile']].copy()
imd_leeds = imd_leeds.rename(columns={'LSOA Code': 'LSOA21CD', 'IMD Decile': 'imd_decile'})


In [28]:
pop_leeds

,LSOA21CD,population
27932,E01011264,1296.0
27933,E01011265,1967.0
27934,E01011266,2652.0
27935,E01011267,1696.0
27936,E01011268,1416.0
...,...,...
28415,E01035050,1628.0
28416,E01035051,1561.0
28417,E01035052,1414.0
28418,E01035053,1722.0


In [29]:
imd_leeds

,LSOA21CD,imd_decile
10719,E01011264,6
10720,E01011265,8
10721,E01011266,10
10722,E01011267,4
10723,E01011268,3
...,...,...
33044,E01035050,5
33045,E01035051,9
33046,E01035052,4
33047,E01035053,7


In [30]:
# Align both to node order
eval_df = pd.DataFrame({'LSOA21CD': list(all_lsoas_leeds)})
eval_df = eval_df.merge(pop_leeds, on='LSOA21CD', how='left')
eval_df = eval_df.merge(imd_leeds, on='LSOA21CD', how='left')
eval_df = eval_df.fillna(0)

population = eval_df['population'].values
imd_decile = eval_df['imd_decile'].values.astype(int)

print(f"Population matched: {(population > 0).sum()} / {len(population)}")
print(f"IMD matched:        {(imd_decile > 0).sum()} / {len(imd_decile)}")
print(f"Population range:   {population.min():.0f} - {population.max():.0f}")
print(f"IMD decile range:   {imd_decile.min()} - {imd_decile.max()}")

Population matched: 488 / 488
IMD matched:        488 / 488
Population range:   988 - 4228
IMD decile range:   1 - 10


## GRAVITY MODEL

In [31]:
# Population
X_train, X_test, y_train, y_test = train_test_split(
    poi_norm, population, test_size=0.2, random_state=42)
grav_pop_r2 = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))
print(f"\nGravity; Population R²: {grav_pop_r2:.4f}")

# IMD
X_train, X_test, y_train, y_test = train_test_split(
    poi_norm, imd_decile, test_size=0.2, random_state=42)
grav_imd_acc = accuracy_score(y_test,
    RandomForestClassifier(n_estimators=100, random_state=42)
    .fit(X_train, y_train).predict(X_test))
print(f"Gravity; IMD Accuracy:  {grav_imd_acc:.4f}")


Gravity; Population R²: 0.0811
Gravity; IMD Accuracy:  0.1837


previous result 
Gravity; Population R²: 0.0731
Gravity; IMD Accuracy:  0.1735

## ML

In [32]:
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### Build node index and PyG graph

In [33]:
# Node index 
lsoa_to_idx = {lsoa: i for i, lsoa in enumerate(all_lsoas_leeds)}

leeds_od['src'] = leeds_od['origin_lsoa'].map(lsoa_to_idx)
leeds_od['dst'] = leeds_od['workplace_lsoa'].map(lsoa_to_idx)

# Normalise POI for DGI 
x = torch.tensor(feature_df[poi_cols].values, dtype=torch.float)
x = torch.log1p(x)
x = (x - x.mean(dim=0)) / (x.std(dim=0) + 1e-8)

# PyG graph 
edge_index  = torch.tensor(leeds_od[['src', 'dst']].values.T, dtype=torch.long)
edge_weight = torch.tensor(leeds_od['flow_log'].values, dtype=torch.float)
data        = Data(x=x, edge_index=edge_index, edge_weight=edge_weight)

print(f"Nodes:         {data.num_nodes}")
print(f"Edges:         {data.num_edges}")
print(f"Node features: {data.x.shape}")
print(f"x range:       [{data.x.min():.3f}, {data.x.max():.3f}]")
print(f"Any NaN:       {torch.isnan(data.x).any()}")

Nodes:         488
Edges:         54907
Node features: torch.Size([488, 11])
x range:       [-1.142, 6.574]
Any NaN:       False


### train DGI

In [34]:

# Encoder 
class DGIEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.prelu = torch.nn.PReLU()

    def forward(self, x, edge_index, edge_weight=None):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        x = self.prelu(x)
        return x

def corruption(x, edge_index, edge_weight=None):
    return x[torch.randperm(x.size(0))], edge_index, edge_weight

encoder = DGIEncoder(in_channels=data.x.shape[1],
                     hidden_channels=128, out_channels=64)

dgi = DeepGraphInfomax(
    hidden_channels=64,
    encoder=encoder,
    summary=lambda z, *args, **kwargs: torch.sigmoid(z.mean(dim=0)),
    corruption=corruption
)

optimizer = torch.optim.Adam(dgi.parameters(), lr=0.001)

# Training 
dgi.train()
for epoch in range(300):
    optimizer.zero_grad()
    pos_z, neg_z, summary = dgi(data.x, data.edge_index, data.edge_weight)
    loss = dgi.loss(pos_z, neg_z, summary)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

# Extract embeddings    
dgi.eval()
with torch.no_grad():
    embeddings = encoder(data.x, data.edge_index, data.edge_weight)

emb = embeddings.numpy()
print(f"\nEmbeddings: {emb.shape}")
print(f"Range:      [{emb.min():.3f}, {emb.max():.3f}]")
print(f"Std:        {emb.std():.4f}")

Epoch   0 | Loss: 1.3601
Epoch  50 | Loss: 0.0580
Epoch 100 | Loss: 0.1147
Epoch 150 | Loss: 0.0236
Epoch 200 | Loss: 0.0121
Epoch 250 | Loss: 0.0072

Embeddings: (488, 64)
Range:      [-0.327, 2.590]
Std:        0.2587


#### DGI: population and IMD prediction

In [43]:
# Population
X_train, X_test, y_train, y_test = train_test_split(
    emb, population, test_size=0.2, random_state=42)
dgi_pop_r2 = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))
print(f"DGI  Population R²: {dgi_pop_r2:.4f}")

# IMD
X_train, X_test, y_train, y_test = train_test_split(
    emb, imd_decile, test_size=0.2, random_state=42)
dgi_imd_acc = accuracy_score(y_test,
    RandomForestClassifier(n_estimators=100, random_state=42)
    .fit(X_train, y_train).predict(X_test))
print(f"DGI  IMD Accuracy:  {dgi_imd_acc:.4f}")

DGI  Population R²: -0.0741
DGI  IMD Accuracy:  0.2041


previous result - population: 0.0831

In [36]:
lsoa_centroids = pd.read_csv('./data/LSOA_PopCentroids_EW_2021_V4_-1224685956027462334.csv')

In [37]:
lsoa_centroids

,FID,LSOA21CD,GlobalID,GlobalID_2,x,y
0,1,E01000001,38ad8fe8-682a-4533-b51a-16c9ca366294,dd315ba1-4937-466d-8c6a-cf5d42615b0a,532181.7642,181792.2818
1,2,E01018945,0681c038-f7dd-4a2e-83c1-c8d3b32443f6,3ae37539-20db-4f78-9537-61bfddacfce5,202699.4208,65668.0221
2,3,E01000120,e5ffc70b-0e24-40a9-afe8-13da1ec4032e,78bebe76-7b81-4351-bc3b-0b3ac11ca8e5,528823.5718,194238.0466
3,4,E01031487,17c7f13f-1a6a-45c5-a5cc-2c50e725c799,e79a12bf-45a3-4649-b1fe-8c5a93dca04a,486282.4085,106971.9143
4,5,E01021013,f55e8508-e975-4025-896c-a270de41fc79,054afc3d-7ce7-42fe-acf2-00789f24448b,583256.5228,111412.9755
...,...,...,...,...,...,...
35667,35668,E01023639,acc5933e-f958-4d5e-9d7c-bf80d3b3a8e4,7e1d7541-259b-42fb-bc83-7329f00e5267,522759.6016,231115.0000
35668,35669,E01035505,8cd1a7c3-774b-445d-9586-0d3cd2ec8816,fc470564-4157-476f-b15e-627fd8e8a6e5,455489.7686,339157.6943
35669,35670,E01012119,3e2e1865-4904-4bae-be77-16dc09da2363,62dfbe5d-e5a2-4c65-8194-89d88cb70cda,460894.7087,516424.3090
35670,35671,E01026708,02dfb6da-68f7-4b30-a2ab-ca024fcf491e,0c0a9816-806a-44b5-9022-ba336e3d54ba,564232.3414,323205.5021


In [38]:
centroid_leeds = lsoa_centroids[lsoa_centroids['LSOA21CD'].isin(all_lsoas_leeds)].copy()
coord_df = pd.DataFrame({'LSOA21CD': list(all_lsoas_leeds)})
coord_df = coord_df.merge(centroid_leeds[['LSOA21CD', 'x', 'y']],
                           on='LSOA21CD', how='left')

print(f"Centroids matched: {coord_df['x'].notna().sum()} / {len(all_lsoas_leeds)}")

Centroids matched: 488 / 488


In [39]:
coord_df.head()

,LSOA21CD,x,y
0,E01011264,421011.7597,441621.2411
1,E01011265,419074.8324,442053.6557
2,E01011266,417665.4848,442750.9094
3,E01011267,419707.0994,441948.1695
4,E01011268,420266.9498,441817.4392


In [40]:
leeds_od.head()

,origin_lsoa,workplace_lsoa,flow,flow_log,src,dst
0,E01011264,E01011265,17,2.890372,0,1
1,E01011264,E01011266,1,0.693147,0,2
2,E01011264,E01011267,1,0.693147,0,3
3,E01011264,E01011268,5,1.791759,0,4
4,E01011264,E01011269,3,1.386294,0,5


In [44]:
coords  = torch.tensor(coord_df[['x', 'y']].values, dtype=torch.float)
src_idx = torch.tensor(leeds_od['src'].values, dtype=torch.long)
dst_idx = torch.tensor(leeds_od['dst'].values, dtype=torch.long)

dist     = torch.sqrt(((coords[src_idx] - coords[dst_idx]) ** 2).sum(dim=1))
dist_log = torch.log1p(dist)
dist_log = (dist_log - dist_log.mean()) / (dist_log.std() + 1e-8)

od_features = torch.cat([
    embeddings[src_idx],
    embeddings[dst_idx],
    dist_log.unsqueeze(1)
], dim=1).numpy()

flow_target = leeds_od['flow_log'].values

X_train, X_test, y_train, y_test = train_test_split(
    od_features, flow_target, test_size=0.2, random_state=42)

flow_r2     = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))


print(f"DGI + Distance  Flow R²:  {flow_r2:.4f}")

DGI + Distance  Flow R²:  0.4774


In [42]:
results_all = pd.DataFrame([
    ("Population",      "Gravity Model (POI)", "R²",       grav_pop_r2),
    ("Population",      "DGI Emb + POI",      "R²",       dgi_pop_r2    ),
    ("IMD Decile",      "Gravity Model (POI)", "Accuracy", grav_imd_acc ),
    ("IMD Decile",      "DGI Emb + POI",      "Accuracy", dgi_imd_acc   ),
    ("Flow Prediction", "DGI + Distance",      "R²",       flow_r2      ),
], columns=["Task", "Model", "Metric", "Leeds"])

print(results_all.to_string(index=False))

           Task               Model   Metric     Leeds
     Population Gravity Model (POI)       R²  0.081076
     Population       DGI Emb + POI       R² -0.074074
     IMD Decile Gravity Model (POI) Accuracy  0.183673
     IMD Decile       DGI Emb + POI Accuracy  0.204082
Flow Prediction      DGI + Distance       R²  0.477428
